In [ ]:
# Reverse Product Search

## Task 3 – Text-Based Product Search using CLIP

#This notebook demonstrates how to search products using natural language queries.
#The query is converted into an embedding using CLIP and compared with product embeddings using cosine similarity.

In [ ]:
# Uncomment if running for the first time

#!pip install -q transformers torch torchvision pillow scikit-learn pandas numpy

In [ ]:
import numpy as np
import pandas as pd
import torch

from sklearn.metrics.pairwise import cosine_similarity

from transformers import (
    CLIPProcessor,
    CLIPModel,
)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

In [ ]:
# Load CLIP model and processor

processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
)

model = model.to(device)
model.eval()

print("Model loaded successfully!")
print(type(model))

In [ ]:
products = [
    {"id":"1","name":"Men's Blue Casual Shirt","category":"Shirts","price":29.99},
    {"id":"2","name":"Blue Checked Shirt","category":"Shirts","price":34.99},
    {"id":"3","name":"Slim Fit Blue Shirt","category":"Shirts","price":39.99},
    {"id":"4","name":"Red Formal Shirt","category":"Shirts","price":44.99},
    {"id":"5","name":"White Casual Shirt","category":"Shirts","price":27.99},
    {"id":"6","name":"Black Denim Jacket","category":"Jackets","price":79.99},
    {"id":"7","name":"Blue Summer Dress","category":"Dresses","price":54.99},
    {"id":"8","name":"Red Casual Pants","category":"Pants","price":49.99},
    {"id":"9","name":"Green Hoodie","category":"Hoodies","price":59.99},
    {"id":"10","name":"Brown Leather Belt","category":"Accessories","price":24.99},
]

products_df = pd.DataFrame(products)
products_df

In [ ]:
def encode_text(text):
    """
    Encode a text query into a normalized CLIP embedding.
    """

    inputs = processor(
        text=[text],
        return_tensors="pt",
        padding=True,
        truncation=True,
    )

    # Move tensors to GPU/CPU
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        embedding = model.get_text_features(**inputs)

    # Normalize
    embedding = embedding / embedding.norm(p=2, dim=-1, keepdim=True)

    return embedding.squeeze().cpu().numpy()

In [ ]:
descriptions = {
    "1": "men blue casual shirt",
    "2": "blue checked shirt",
    "3": "slim fit blue shirt",
    "4": "red formal shirt",
    "5": "white casual shirt",
    "6": "black denim jacket",
    "7": "blue summer dress",
    "8": "red casual pants",
    "9": "green hoodie",
    "10": "brown leather belt",
}

image_embeddings = {}
meta = []

for product in products:

    pid = product["id"]

    image_embeddings[pid] = encode_text(descriptions[pid])

    meta.append(product)

print("Embeddings Created:", len(image_embeddings))

In [ ]:
def search_products(query, top_k=3):

    query_embedding = encode_text(query)

    product_ids = list(image_embeddings.keys())

    vectors = np.array(
        [image_embeddings[pid] for pid in product_ids]
    )

    similarities = cosine_similarity(
        [query_embedding],
        vectors
    )[0]

    sorted_idx = similarities.argsort()[::-1][:top_k]

    results = []

    for idx in sorted_idx:

        pid = product_ids[idx]

        product = next(
            p for p in meta if p["id"] == pid
        )

        results.append({
            "id": pid,
            "name": product["name"],
            "category": product["category"],
            "price": product["price"],
            "score": float(similarities[idx])
        })

    return results

In [ ]:
def display_results(query, results):

    print("=" * 60)
    print("Search Query:", query)
    print("=" * 60)

    for i, item in enumerate(results, start=1):

        print(f"\nRank {i}")
        print("Product :", item["name"])
        print("Category:", item["category"])
        print("Price   : $", item["price"])
        print("Score   :", round(item["score"], 4))

In [ ]:
query = "blue casual shirt"

results = search_products(query)

display_results(query, results)

results

In [ ]:
while True:

    query = input("Enter product description (or quit): ")

    if query.lower()=="quit":
        break

    results = search_products(query)

    display_results(query,results)